# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

SEED = 42
tf.keras.utils.set_random_seed(SEED)  # seeds python, numpy and tensorflow in one call

# 1. Read the dataset
df = pd.read_csv("tic-tac-toe.csv")

# 2. Inspect
print(df.shape)
display(df.head())
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print(df["class"].value_counts(normalize=True).round(3))
display(df.describe())

In [ ]:
# 3. Convert categoricals to numbers. Map x -> 1, o -> -1, b (blank) -> 0.
#    This encoding is symmetric around 0 and already lies in [-1, 1], so it doubles
#    as the normalization step (4/5) - no scaler needed, and it keeps the meaning
#    "X's mark vs O's mark vs empty" instead of an arbitrary label-encoder order.
features = df.columns[:-1]
X = df[features].replace({"x": 1, "o": -1, "b": 0}).to_numpy(dtype="float32")
y = df["class"].astype("int32").to_numpy()   # True -> 1, False -> 0

print(X.shape, y.shape, X.min(), X.max())

## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [ ]:
# 1. Split (stratified so both sets keep the ~65/35 class ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

def build_model(hidden=(64, 64), lr=1e-3):
    model = tf.keras.Sequential([tf.keras.Input(shape=(X.shape[1],))])
    for units in hidden:
        model.add(tf.keras.layers.Dense(units, activation="relu"))
    model.add(tf.keras.layers.Dense(2, activation="softmax"))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

# 2-4. Baseline model with default Adam settings
model = build_model(hidden=(32, 16))
model.summary()

# 5. Fit
history = model.fit(X_train, y_train, epochs=20, batch_size=32,
                    validation_split=0.1, verbose=0)

# 6. Evaluate
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Baseline  -> test loss: {loss:.4f} | test accuracy: {acc:.4f}")

# 7. Save (Keras 3 requires the .keras extension)
model.save("tic-tac-toe.model.keras")

## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [ ]:
loaded = tf.keras.models.load_model("tic-tac-toe.model.keras")

rng = np.random.default_rng(SEED)
idx = rng.choice(len(X_test), size=10, replace=False)

pred = loaded.predict(X_test[idx], verbose=0).argmax(axis=1)  # one batched call

results = pd.DataFrame({
    "board": ["".join({1: "x", -1: "o", 0: "."}[v] for v in row) for row in X_test[idx].astype(int)],
    "actual": y_test[idx].astype(bool),
    "predicted": pred.astype(bool),
})
results["correct"] = results["actual"] == results["predicted"]
display(results)
print(f"{results['correct'].sum()}/{len(results)} correct")

## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [ ]:
# Train longer, with a deeper/wider net and a tuned learning rate.
# EarlyStopping stops when validation loss stops improving and restores the best
# weights, so we can set a generous epoch budget without overfitting or wasting time.
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=20, restore_best_weights=True
)

configs = {
    "baseline (32,16), lr=1e-3, 20 ep": dict(hidden=(32, 16), lr=1e-3, epochs=20, cb=[]),
    "more epochs (32,16), lr=1e-3":     dict(hidden=(32, 16), lr=1e-3, epochs=300, cb=[early_stop]),
    "higher lr (32,16), lr=5e-3":       dict(hidden=(32, 16), lr=5e-3, epochs=300, cb=[early_stop]),
    "deeper (128,64,32), lr=5e-3":      dict(hidden=(128, 64, 32), lr=5e-3, epochs=300, cb=[early_stop]),
}

rows, models = [], {}
for name, c in configs.items():
    tf.keras.utils.set_random_seed(SEED)
    m = build_model(c["hidden"], c["lr"])
    h = m.fit(X_train, y_train, epochs=c["epochs"], batch_size=32,
              validation_split=0.1, callbacks=c["cb"], verbose=0)
    loss, acc = m.evaluate(X_test, y_test, verbose=0)
    rows.append({"model": name, "epochs_run": len(h.history["loss"]),
                 "test_loss": round(loss, 4), "test_accuracy": round(acc, 4)})
    models[name] = m

comparison = pd.DataFrame(rows).set_index("model")
display(comparison)

best_name = comparison["test_loss"].idxmin()
print("Best:", best_name)
models[best_name].save("tic-tac-toe.model.keras")  # overwrite with the improved model

**Which approach(es) did you find helpful to improve your model performance?**

**Answer:**

- **More epochs (with EarlyStopping)** helped the most. 20 epochs isn't enough for this dataset. The baseline is still underfitting and its loss is well above 0.1. Training for longer lets the network learn the winning-line patterns. EarlyStopping (`restore_best_weights=True`) stops once validation loss stops improving, so the larger epoch budget doesn't cause overfitting.
- **A higher learning rate (5e-3 instead of 1e-3)** helped it converge faster: it reaches low loss in fewer epochs. If you push it much higher, training gets unstable.
- **More/wider layers** gave a smaller gain. The problem is small (9 inputs, 8 possible winning lines), so a (32, 16) network can already fit it once it trains long enough. A deeper (128, 64, 32) network can only help a little, and if it doesn't beat the smaller one on test loss, the extra layers aren't needed.

Overall, training long enough with a sensible learning rate mattered more than model size. See the `comparison` table above for the exact test loss/accuracy of each configuration.
